# Cobblemon Atlas｜烈空座物理攻擊鏡頭 V3

同時間軸比較固定鏡頭、V2 與限制跳動的 V3。全程使用候選資料夾，**不覆蓋原有 WebP 或網站**。

In [ ]:
from pathlib import Path
import subprocess,sys,zipfile,shutil,json
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Cobblemon_Atlas')
WORK=Path('/content/cobblemon-bedrock')
SITE=Path('/content/cobblemon-atlas')
BRANCH='feat/animation-manifest-scanner-phase1'
assert ROOT.is_dir()

In [ ]:
if not (SITE/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','--branch',BRANCH,'https://github.com/hwh0410/cobblemon-atlas.git',str(SITE)],check=True)
subprocess.run(['git','-C',str(SITE),'sparse-checkout','set','tools','notebooks'],check=True)
subprocess.run(['git','-C',str(SITE),'pull','--ff-only','origin',BRANCH],check=True)
print('Git:',subprocess.check_output(['git','-C',str(SITE),'rev-parse','--short','HEAD'],text=True).strip())

In [ ]:
WORK.mkdir(parents=True,exist_ok=True)
for name in ('merged_pack','development_resource_packs'):
    expected=WORK/name
    if not expected.is_dir():
        archive=ROOT/'source_packs'/(name+'.zip')
        assert archive.is_file(),archive
        with zipfile.ZipFile(archive) as z:
            for item in z.infolist():
                assert (WORK/item.filename).resolve().is_relative_to(WORK.resolve()),'Unsafe zip path'
            z.extractall(WORK)
    assert expected.is_dir(),expected
(WORK/'tools').mkdir(exist_ok=True)
shutil.copy2(ROOT/'source_packs/render_portraits.py',WORK/'tools/render_portraits.py')
print('Source pack ready')

In [ ]:
resolved=ROOT/'reports/resolver-validation/resolved-forms.json'
out=ROOT/'reports/resolver-validation/motion-camera-v3'
subprocess.run([sys.executable,str(SITE/'tools/render_rayquaza_motion_camera_v3.py'),'--repo',str(WORK),'--resolved',str(resolved),'--out',str(out)],check=True)
r=json.loads((out/'physical-attack-v3-report.json').read_text())
for key in ('max_jump_fixed_px','max_jump_v2_px','max_jump_v3_px','min_margin_px','unsafe_frames','decoded_frames','decoded_duration_ms','status'):
    print(key,r.get(key))
print('Report:',out/'physical-attack-v3-report.json')